# Experiment: AKI Model Development and Evaluation

This notebook is organized to satisfy the model-development writeup requirements:

1. exploratory data analysis with tables and figures
2. data preprocessing for missingness and feature engineering
3. subject-level data splitting for model development
4. performance evaluation with AUROC, PR-AUC, and calibration curves for at least two models
5. interpretation of performance across models and splits


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET_COL = 'future_aki_24h'
ID_COLS = ['subject_id', 'hadm_id', 'stay_id']
TIME_COLS = [
    'icu_intime',
    'anchor_time',
    'obs_window_start',
    'obs_window_end',
    'pred_window_start',
    'pred_window_end',
]
LEAKAGE_COLS = [
    'hospital_expire_flag',
    'los_icu',
    'los_hospital',
    'icu_outtime',
    'dischtime',
    'aki_onset_time',
    'aki_onset_stage',
    'aki_stage_max',
    'has_aki_anytime',
    'eligible_for_prediction',
]
MISSINGNESS_THRESHOLD = 95.0
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_DEV = 0.25
RANDOM_STATE = 42

CANDIDATE_INPUTS = [
    Path('outputs/mimic_aki_cohort_cleaned.csv'),
    Path('notebook/outputs/mimic_aki_cohort_cleaned.csv'),
    Path('../outputs/mimic_aki_cohort_cleaned.csv'),
    Path('/content/BIS568_Final_Project/notebook/outputs/mimic_aki_cohort_cleaned.csv'),
    Path('/content/BIS568_Final_Project/outputs/mimic_aki_cohort_cleaned.csv'),
]


def resolve_input_csv():
    for candidate in CANDIDATE_INPUTS:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate mimic_aki_cohort_cleaned.csv in the expected outputs directories.')


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


INPUT_CSV = resolve_input_csv()
ARTIFACT_DIR = INPUT_CSV.parent / 'model_development_outputs'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print({'input_csv': str(INPUT_CSV), 'artifact_dir': str(ARTIFACT_DIR)})


{'input_csv': 'outputs/mimic_aki_cohort_cleaned.csv', 'artifact_dir': 'outputs/model_development_outputs'}


## Section 1: Setup & Data Loading

Load the cleaned AKI cohort and keep the notebook structure simple and report-friendly, similar to PA2.


In [2]:
df = pd.read_csv(INPUT_CSV)
for col in TIME_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

if df[TARGET_COL].isna().any():
    df = df[df[TARGET_COL].notna()].copy()

df[TARGET_COL] = df[TARGET_COL].astype(int)

print(f'Cleaned data shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print('Target distribution:')
print(df[TARGET_COL].value_counts(dropna=False).sort_index())
print('Unique subjects:', df['subject_id'].nunique())
df.head()


Cleaned data shape: 29,274 rows x 156 columns
Target distribution:
future_aki_24h
0    12156
1    17118
Name: count, dtype: int64
Unique subjects: 29274


,subject_id,hadm_id,stay_id,icu_intime,anchor_time,obs_window_start,obs_window_end,pred_window_start,pred_window_end,future_aki_24h,...,spo2_last_6h,spo2_min_6h,spo2_max_6h,spo2_mean_6h,spo2_count_6h,urine_total_6h,urine_count_6h,weight_used_for_uo_norm,urine_rate_6h,oliguria_like_6h
0,10001725,25563031,31205490,2110-04-11 15:52:22,2110-04-11 21:52:22,2110-04-11 15:52:22,2110-04-11 21:52:22,2110-04-11 21:52:22,2110-04-12 21:52:22,0,...,98.0,96.0,100.0,98.833333,6.0,300.0,1.0,72.2,0.692521,0.0
1,10001884,26184834,37510196,2131-01-11 04:20:05,2131-01-11 10:20:05,2131-01-11 04:20:05,2131-01-11 10:20:05,2131-01-11 10:20:05,2131-01-12 10:20:05,0,...,100.0,90.0,100.0,98.000000,7.0,625.0,4.0,65.0,1.602564,0.0
2,10002013,23581541,39060235,2160-05-18 10:00:53,2160-05-18 16:00:53,2160-05-18 10:00:53,2160-05-18 16:00:53,2160-05-18 16:00:53,2160-05-19 16:00:53,1,...,100.0,100.0,100.0,100.000000,2.0,280.0,3.0,96.0,0.486111,1.0
3,10002155,23822395,33685454,2129-08-04 12:45:00,2129-08-04 18:45:00,2129-08-04 12:45:00,2129-08-04 18:45:00,2129-08-04 18:45:00,2129-08-05 18:45:00,1,...,92.0,92.0,97.0,93.428571,7.0,250.0,1.0,53.0,0.786164,0.0
4,10002348,22725460,32610785,2112-11-30 23:24:00,2112-12-01 05:24:00,2112-11-30 23:24:00,2112-12-01 05:24:00,2112-12-01 05:24:00,2112-12-02 05:24:00,0,...,95.0,93.0,97.0,94.571429,7.0,50.0,1.0,41.6,0.200321,1.0


---
## Section 2: EDA — Table 1 (Cohort Demographics)

Keep the EDA lightweight: one demographics table and a couple of simple figures.


In [3]:
demo_summary = pd.DataFrame({
    'metric': [
        'rows',
        'unique_subjects',
        'positive_labels',
        'positive_rate_pct',
        'median_age',
        'female_pct',
    ],
    'overall': [
        len(df),
        df['subject_id'].nunique(),
        int(df[TARGET_COL].sum()),
        round(df[TARGET_COL].mean() * 100, 2),
        round(df['admission_age'].median(), 2) if 'admission_age' in df.columns else np.nan,
        round((df['gender'].astype(str).str.upper() == 'F').mean() * 100, 2) if 'gender' in df.columns else np.nan,
    ],
    'label_0': [
        len(df[df[TARGET_COL] == 0]),
        df.loc[df[TARGET_COL] == 0, 'subject_id'].nunique(),
        int(df.loc[df[TARGET_COL] == 0, TARGET_COL].sum()),
        round(df.loc[df[TARGET_COL] == 0, TARGET_COL].mean() * 100, 2),
        round(df.loc[df[TARGET_COL] == 0, 'admission_age'].median(), 2) if 'admission_age' in df.columns else np.nan,
        round((df.loc[df[TARGET_COL] == 0, 'gender'].astype(str).str.upper() == 'F').mean() * 100, 2) if 'gender' in df.columns else np.nan,
    ],
    'label_1': [
        len(df[df[TARGET_COL] == 1]),
        df.loc[df[TARGET_COL] == 1, 'subject_id'].nunique(),
        int(df.loc[df[TARGET_COL] == 1, TARGET_COL].sum()),
        round(df.loc[df[TARGET_COL] == 1, TARGET_COL].mean() * 100, 2),
        round(df.loc[df[TARGET_COL] == 1, 'admission_age'].median(), 2) if 'admission_age' in df.columns else np.nan,
        round((df.loc[df[TARGET_COL] == 1, 'gender'].astype(str).str.upper() == 'F').mean() * 100, 2) if 'gender' in df.columns else np.nan,
    ],
})

gender_table = pd.crosstab(df['gender'], df[TARGET_COL], normalize='columns').round(3) if 'gender' in df.columns else None
race_table = pd.crosstab(df['race'], df[TARGET_COL], normalize='columns').round(3).head(10) if 'race' in df.columns else None

display(demo_summary)
if gender_table is not None:
    display(gender_table)
if race_table is not None:
    display(race_table)

demo_summary.to_csv(ARTIFACT_DIR / 'cohort_demographics_summary.csv', index=False)
if gender_table is not None:
    gender_table.to_csv(ARTIFACT_DIR / 'gender_by_target.csv')
if race_table is not None:
    race_table.to_csv(ARTIFACT_DIR / 'race_by_target_top10.csv')


,metric,overall,label_0,label_1
0,rows,29274.00,12156.00,17118.00
1,unique_subjects,29274.00,12156.00,17118.00
2,positive_labels,17118.00,0.00,17118.00
3,positive_rate_pct,58.48,0.00,100.00
4,median_age,67.00,63.00,69.00
5,female_pct,44.28,45.88,43.14


future_aki_24h,0,1
gender,,
F,0.459,0.431
M,0.541,0.569


future_aki_24h,0,1
race,,
AMERICAN INDIAN/ALASKA NATIVE,0.002,0.002
ASIAN,0.015,0.009
ASIAN - ASIAN INDIAN,0.003,0.002
ASIAN - CHINESE,0.013,0.007
ASIAN - KOREAN,0.002,0.001
ASIAN - SOUTH EAST ASIAN,0.004,0.002
BLACK/AFRICAN,0.005,0.003
BLACK/AFRICAN AMERICAN,0.065,0.066
BLACK/CAPE VERDEAN,0.005,0.005


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_counts = df[TARGET_COL].value_counts().sort_index()
axes[0].bar(['No AKI in 24h', 'Future AKI in 24h'], label_counts.values, color=['#4C78A8', '#E45756'])
axes[0].set_title('Target distribution')
axes[0].set_ylabel('Number of ICU stays')

if 'admission_age' in df.columns:
    age_groups = [df.loc[df[TARGET_COL] == 0, 'admission_age'].dropna(), df.loc[df[TARGET_COL] == 1, 'admission_age'].dropna()]
    axes[1].boxplot(age_groups, labels=['Label 0', 'Label 1'])
    axes[1].set_title('Admission age by target')
    axes[1].set_ylabel('Admission age')
else:
    axes[1].axis('off')
    axes[1].text(0.1, 0.5, 'admission_age not available', fontsize=11)

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'simple_eda_figures.png', dpi=200, bbox_inches='tight')
plt.show()


---
## Section 3: Feature Engineering

Keep the feature engineering code in one cell so it is easy to explain in the report.


In [ ]:
def add_engineered_features(frame):
    engineered = frame.copy()

    if {'bun_last_6h', 'creatinine_last_6h'}.issubset(engineered.columns):
        engineered['bun_creatinine_ratio_last_6h'] = engineered['bun_last_6h'] / engineered['creatinine_last_6h'].replace(0, np.nan)

    if {'sbp_last_6h', 'dbp_last_6h'}.issubset(engineered.columns):
        engineered['pulse_pressure_last_6h'] = engineered['sbp_last_6h'] - engineered['dbp_last_6h']

    if {'heart_rate_last_6h', 'sbp_last_6h'}.issubset(engineered.columns):
        engineered['shock_index_last_6h'] = engineered['heart_rate_last_6h'] / engineered['sbp_last_6h'].replace(0, np.nan)

    if {'urine_total_6h', 'weight_used_for_uo_norm'}.issubset(engineered.columns):
        engineered['urine_ml_per_kg_6h'] = engineered['urine_total_6h'] / engineered['weight_used_for_uo_norm'].replace(0, np.nan)

    if {'creatinine_last_6h', 'creatinine_first_6h'}.issubset(engineered.columns):
        engineered['creatinine_change_6h'] = engineered['creatinine_last_6h'] - engineered['creatinine_first_6h']

    return engineered


original_columns = set(df.columns)
df = add_engineered_features(df)
engineered_cols = sorted(set(df.columns) - original_columns)

print('Engineered features added:')
print(engineered_cols)


---
## Section 4: Missingness Review

Only print the top 10 missing columns so the notebook stays readable.


In [ ]:
missing_pct = df.isnull().mean() * 100
print(f'Columns with missing values: {(missing_pct > 0).sum()}')
print()
print('Top 10 columns by missing %:')
print(missing_pct[missing_pct > 0].sort_values(ascending=False).head(10).round(2))


---
## Section 5: Data Preprocessing

Remove identifiers, timing columns, explicit leakage columns, very high-missingness columns, and constant columns before modeling.


In [ ]:
required_keep_cols = set(ID_COLS + TIME_COLS + [TARGET_COL])
leakage_cols_present = [col for col in LEAKAGE_COLS if col in df.columns]
missing_pct = df.isna().mean() * 100

high_missing_cols = [
    col for col, pct in missing_pct.items()
    if pct >= MISSINGNESS_THRESHOLD and col not in required_keep_cols
]
constant_cols = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1 and col not in required_keep_cols
]

excluded_feature_cols = sorted(set(ID_COLS + TIME_COLS + leakage_cols_present + high_missing_cols + constant_cols + [TARGET_COL]))
feature_cols = [col for col in df.columns if col not in excluded_feature_cols]

preprocessing_summary = pd.DataFrame({
    'step': [
        'engineered_features_added',
        'high_missing_columns_removed',
        'constant_columns_removed',
        'explicit_leakage_columns_removed',
        'final_feature_count',
    ],
    'count': [
        len(engineered_cols),
        len(high_missing_cols),
        len(constant_cols),
        len(leakage_cols_present),
        len(feature_cols),
    ],
})

display(preprocessing_summary)
print('Removed for high missingness:', high_missing_cols)
print('Removed as constants:', constant_cols)
print('Removed as explicit leakage columns:', leakage_cols_present)
print('First 20 modeling features:', feature_cols[:20])

preprocessing_summary.to_csv(ARTIFACT_DIR / 'preprocessing_summary.csv', index=False)


---
## Section 6: Train / Validation / Test Split

Use subject-level splitting only. The same patient should never appear in multiple partitions.


In [ ]:
groups = df['subject_id']
outer_split = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
dev_idx, test_idx = next(outer_split.split(df, df[TARGET_COL], groups=groups))

dev_df = df.iloc[dev_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

inner_split = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE_WITHIN_DEV, random_state=RANDOM_STATE)
train_idx, val_idx = next(inner_split.split(dev_df, dev_df[TARGET_COL], groups=dev_df['subject_id']))

train_df = dev_df.iloc[train_idx].reset_index(drop=True)
val_df = dev_df.iloc[val_idx].reset_index(drop=True)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_df), len(val_df), len(test_df)],
    'unique_subjects': [
        train_df['subject_id'].nunique(),
        val_df['subject_id'].nunique(),
        test_df['subject_id'].nunique(),
    ],
    'positive_rate_pct': [
        round(train_df[TARGET_COL].mean() * 100, 2),
        round(val_df[TARGET_COL].mean() * 100, 2),
        round(test_df[TARGET_COL].mean() * 100, 2),
    ],
})

display(split_summary)
print('Overlap train/validation:', len(set(train_df['subject_id']).intersection(set(val_df['subject_id']))))
print('Overlap train/test:', len(set(train_df['subject_id']).intersection(set(test_df['subject_id']))))
print('Overlap validation/test:', len(set(val_df['subject_id']).intersection(set(test_df['subject_id']))))

split_summary.to_csv(ARTIFACT_DIR / 'split_summary.csv', index=False)


## Section 7: Model Fitting Setup

Two models are developed on the same feature set:

- **Logistic regression**: a linear baseline with standardized numeric features
- **Random forest**: a nonlinear tree-based baseline with the same train / validation / test split

This keeps the comparison focused on architecture rather than changing the data definition.


In [7]:
train_X = train_df[feature_cols].copy()
val_X = val_df[feature_cols].copy()
test_X = test_df[feature_cols].copy()
train_y = train_df[TARGET_COL].astype(int)
val_y = val_df[TARGET_COL].astype(int)
test_y = test_df[TARGET_COL].astype(int)

numeric_cols = train_X.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols = [col for col in feature_cols if col not in numeric_cols]

logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', make_one_hot_encoder()),
            ]),
            categorical_cols,
        ),
    ],
    remainder='drop',
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            numeric_cols,
        ),
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', make_one_hot_encoder()),
            ]),
            categorical_cols,
        ),
    ],
    remainder='drop',
)

models = {
    'Logistic Regression': Pipeline([
        ('preprocess', logistic_preprocessor),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ]),
    'Random Forest': Pipeline([
        ('preprocess', rf_preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight='balanced_subsample',
        )),
    ]),
}

print({'numeric_features': len(numeric_cols), 'categorical_features': len(categorical_cols)})


{'numeric_features': 144, 'categorical_features': 7}


## Section 8: Model Development and Evaluation Metrics

Each model is trained on the training split and then evaluated on both the validation split and the held-out test split. The core metrics are:

- AUROC
- PR-AUC
- Brier score (added as a compact calibration summary)

Calibration curves are plotted separately in the next section.


In [8]:
def evaluate_predictions(y_true, proba):
    return {
        'auroc': roc_auc_score(y_true, proba),
        'pr_auc': average_precision_score(y_true, proba),
        'brier_score': brier_score_loss(y_true, proba),
    }


results = []
prediction_store: dict[str, dict[str, np.ndarray]] = {}

for model_name, pipeline in models.items():
    fitted = clone(pipeline)
    fitted.fit(train_X, train_y)
    prediction_store[model_name] = {}

    for split_name, X_split, y_split in [
        ('validation', val_X, val_y),
        ('test', test_X, test_y),
    ]:
        proba = fitted.predict_proba(X_split)[:, 1]
        prediction_store[model_name][split_name] = proba
        metrics = evaluate_predictions(y_split, proba)
        results.append(
            {
                'model': model_name,
                'split': split_name,
                **metrics,
            }
        )

metrics_df = pd.DataFrame(results).sort_values(['split', 'pr_auc'], ascending=[True, False])
display(metrics_df)
metrics_df.to_csv(ARTIFACT_DIR / 'model_metrics.csv', index=False)


,model,split,auroc,pr_auc,brier_score
3,Random Forest,test,0.727285,0.796118,0.205965
1,Logistic Regression,test,0.727904,0.791459,0.210457
0,Logistic Regression,validation,0.740109,0.801460,0.205867
2,Random Forest,validation,0.727601,0.792444,0.206006


## Section 9: Performance Figures

These figures are the main model-evaluation visuals for the report. They show both model comparison and split comparison using the same outcome definition and subject-level split strategy.


In [ ]:
def plot_roc(ax, y_true, prediction_map, split_name):
    from sklearn.metrics import roc_curve

    for model_name, proba in prediction_map.items():
        fpr, tpr, _ = roc_curve(y_true, proba)
        auroc = roc_auc_score(y_true, proba)
        ax.plot(fpr, tpr, linewidth=2, label=f'{model_name} (AUC={auroc:.3f})')
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.5)
    ax.set_title(f'ROC Curve — {split_name.title()} Set')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend()


def plot_pr(ax, y_true, prediction_map, split_name):
    from sklearn.metrics import precision_recall_curve

    for model_name, proba in prediction_map.items():
        precision, recall, _ = precision_recall_curve(y_true, proba)
        pr_auc = average_precision_score(y_true, proba)
        ax.plot(recall, precision, linewidth=2, label=f'{model_name} (PR-AUC={pr_auc:.3f})')
    ax.set_title(f'Precision-Recall Curve — {split_name.title()} Set')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.legend()


def plot_calibration(ax, y_true, prediction_map, split_name):
    for model_name, proba in prediction_map.items():
        frac_pos, mean_pred = calibration_curve(y_true, proba, n_bins=10, strategy='quantile')
        ax.plot(mean_pred, frac_pos, marker='o', linewidth=2, label=model_name)
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.5, label='Perfect calibration')
    ax.set_title(f'Calibration Curve — {split_name.title()} Set')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed event rate')
    ax.legend()


def plot_metric_pair(plot_func, y_left, preds_left, left_name, y_right, preds_right, right_name, suptitle, output_name):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    plot_func(axes[0], y_left, preds_left, left_name)
    plot_func(axes[1], y_right, preds_right, right_name)
    fig.suptitle(suptitle, fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / output_name, dpi=200, bbox_inches='tight')
    plt.show()


validation_preds = {model: prediction_store[model]['validation'] for model in prediction_store}
test_preds = {model: prediction_store[model]['test'] for model in prediction_store}

plot_metric_pair(
    plot_roc,
    test_y,
    test_preds,
    'test',
    val_y,
    validation_preds,
    'validation',
    'ROC Curves',
    'roc_curves.png',
)

plot_metric_pair(
    plot_pr,
    test_y,
    test_preds,
    'test',
    val_y,
    validation_preds,
    'validation',
    'Precision-Recall Curves',
    'pr_curves.png',
)

plot_metric_pair(
    plot_calibration,
    test_y,
    test_preds,
    'test',
    val_y,
    validation_preds,
    'validation',
    'Calibration Curves',
    'calibration_curves.png',
)


## Section 10: Interpretation of Model Performance

The final cell generates a short text summary that can be adapted into the project report. It compares the best-performing model on the held-out test set, comments on validation-to-test performance shifts, and highlights the calibration tradeoff.


In [12]:
summary_rows = []
for model_name in metrics_df['model'].unique():
    val_row = metrics_df[(metrics_df['model'] == model_name) & (metrics_df['split'] == 'validation')].iloc[0]
    test_row = metrics_df[(metrics_df['model'] == model_name) & (metrics_df['split'] == 'test')].iloc[0]
    summary_rows.append(
        {
            'model': model_name,
            'validation_auroc': val_row['auroc'],
            'test_auroc': test_row['auroc'],
            'validation_pr_auc': val_row['pr_auc'],
            'test_pr_auc': test_row['pr_auc'],
            'validation_brier': val_row['brier_score'],
            'test_brier': test_row['brier_score'],
            'pr_auc_gap_val_to_test': test_row['pr_auc'] - val_row['pr_auc'],
        }
    )

comparison_df = pd.DataFrame(summary_rows).sort_values('test_pr_auc', ascending=False)
display(comparison_df)
comparison_df.to_csv(ARTIFACT_DIR / 'model_comparison_summary.csv', index=False)

best_model = comparison_df.iloc[0]
most_stable = comparison_df.iloc[comparison_df['pr_auc_gap_val_to_test'].abs().argmin()]

interpretation_lines = [
    f"Best held-out test PR-AUC: {best_model['model']} ({best_model['test_pr_auc']:.3f}).",
    f"Best held-out test AUROC: {comparison_df.sort_values('test_auroc', ascending=False).iloc[0]['model']} ({comparison_df['test_auroc'].max():.3f}).",
    f"Most stable validation-to-test PR-AUC gap: {most_stable['model']} ({most_stable['pr_auc_gap_val_to_test']:+.3f}).",
]

for model_name in comparison_df['model']:
    row = comparison_df[comparison_df['model'] == model_name].iloc[0]
    interpretation_lines.append(
        f"{model_name}: validation/test AUROC = {row['validation_auroc']:.3f}/{row['test_auroc']:.3f}, "
        f"validation/test PR-AUC = {row['validation_pr_auc']:.3f}/{row['test_pr_auc']:.3f}, "
        f"validation/test Brier = {row['validation_brier']:.3f}/{row['test_brier']:.3f}."
    )

print(''.join(interpretation_lines))


,model,validation_auroc,test_auroc,validation_pr_auc,test_pr_auc,validation_brier,test_brier,pr_auc_gap_val_to_test
0,Random Forest,0.727601,0.727285,0.792444,0.796118,0.206006,0.205965,0.003674
1,Logistic Regression,0.740109,0.727904,0.801460,0.791459,0.205867,0.210457,-0.010001


Best held-out test PR-AUC: Random Forest (0.796).Best held-out test AUROC: Logistic Regression (0.728).Most stable validation-to-test PR-AUC gap: Random Forest (+0.004).Random Forest: validation/test AUROC = 0.728/0.727, validation/test PR-AUC = 0.792/0.796, validation/test Brier = 0.206/0.206.Logistic Regression: validation/test AUROC = 0.740/0.728, validation/test PR-AUC = 0.801/0.791, validation/test Brier = 0.206/0.210.


## Section 11: Report-Ready Notes

When you write the project report, keep the interpretation tied to these outputs:

- use the EDA tables / figures to motivate data quality and class imbalance
- describe the missingness threshold, imputation rules, and engineered features in the preprocessing section
- report that the split was performed by `subject_id`, not by row
- cite the AUROC, PR-AUC, and calibration plots from the held-out test set when comparing models
- explain any validation-to-test drop as potential overfitting or split sensitivity rather than claiming generalization too aggressively
